# Extract KBStats player data

Fetch all player records from the KBStats API and save timestamped JSON and CSV snapshots in the notebook's working directory.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    KBSTATS_PLAYERS_DIR,
    ensure_directory,
)


In [2]:
# Import the libraries required by this notebook step.
from datetime import datetime
from pathlib import Path
import json

import pandas as pd
import requests


In [3]:
# Set workflow configuration value: API_URL.
API_URL = "https://api.kbstats.de/api/v1/players"
# Set workflow configuration value: OUTPUT_DIR.
OUTPUT_DIR = ensure_directory(KBSTATS_PLAYERS_DIR)

# Local time and UTC offset are included, e.g. 20260810_231500_+0200.
timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%z")
json_path = OUTPUT_DIR / f"kbstats_players_{timestamp}.json"
csv_path = OUTPUT_DIR / f"kbstats_players_{timestamp}.csv"


In [4]:
response = requests.get(
    API_URL,
    headers={"Accept": "application/json"},
    timeout=30,
)
response.raise_for_status()
players = response.json()

# Validate the input before continuing with later processing.
if not isinstance(players, list):
    raise TypeError(
        f"Expected the API to return a list, got {type(players).__name__}."
    )

print(f"Downloaded {len(players):,} player records.")


Downloaded 467 player records.


In [5]:
# Preserve the original nested response in the JSON snapshot.
with json_path.open("w", encoding="utf-8") as file:
    json.dump(players, file, ensure_ascii=False, indent=2)

# Flatten record fields for CSV. Nested lists/dicts remain valid JSON strings.
players_df = pd.json_normalize(players, sep=".")
# Process each available item while preserving the current workflow state.
for column in players_df.columns:
    players_df[column] = players_df[column].map(
        lambda value: (
            json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict))
            else value
        )
    )

# utf-8-sig keeps names with accents readable when opened in Excel.
players_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"JSON saved to: {json_path.resolve()}")
print(f"CSV saved to:  {csv_path.resolve()}")


JSON saved to: C:\kickbase project\outputs\kbstats\players\kbstats_players_20260904_104638_+0200.json
CSV saved to:  C:\kickbase project\outputs\kbstats\players\kbstats_players_20260904_104638_+0200.csv


In [6]:
# Run this step and display the result for inspection.
display(players_df.head())
print(f"Rows: {len(players_df):,} | Columns: {len(players_df.columns):,}")


,id,teamId,firstName,lastName,marketValue,averagePoints,games,gamesPlayed,start11,totalPlaytimeS,...,pointsStdDev,history,startProbability,name,marketChangeToday,marketChange.today,marketChange.yesterday,marketChange.twoDays,marketChange.sevenDaysAvg,marketChange.thirtyDaysAvg
0,173,2,Jonathan,Tah,33630269.0,35.0,1.0,1.0,0.0,693.0,...,NaN,"[{""hasPlayed"": true, ""points"": 35}]",3,Jonathan Tah,-637454.0,-637454.0,-561808.0,-515197.0,-3266044.0,-2432780.0
1,237,2,Manuel,Neuer,14552983.0,145.0,1.0,1.0,1.0,5767.0,...,NaN,"[{""hasPlayed"": true, ""points"": 145}]",1,Manuel Neuer,92910.0,92910.0,131517.0,144356.0,718702.0,2317397.0
2,383,2,Sven,Ulreich,500000.0,NaN,NaN,NaN,NaN,NaN,...,NaN,"[{""hasPlayed"": false, ""points"": null}]",5,Sven Ulreich,0.0,0.0,0.0,0.0,-41527.0,-11636.0
3,1685,2,Joshua,Kimmich,59870453.0,303.0,1.0,1.0,1.0,5767.0,...,NaN,"[{""hasPlayed"": true, ""points"": 303}]",1,Joshua Kimmich,4003.0,4003.0,8506.0,13275.0,-5874.0,559689.0
4,1961,2,Serge,Gnabry,12468509.0,NaN,NaN,NaN,NaN,NaN,...,NaN,"[{""hasPlayed"": false, ""points"": null}]",5,Serge Gnabry,21249.0,21249.0,40444.0,2029.0,-1061834.0,-6140271.0


Rows: 467 | Columns: 27


In [7]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 0 expired timestamped output(s).
